# Analyze stocks with polygon.io and LLM's

Steps:
1. get ticker
2. get daily market summary
3. get moving averages
4. get financials
5. get news sentiments


### Setup

In [76]:

import os
import pandas as pd
from datetime import date
from dotenv import load_dotenv
from pprint import pprint

from openai import OpenAI
from polygon import RESTClient

load_dotenv()

API_KEY=os.getenv("POLYGON_API_KEY")
if API_KEY is None:
    raise ValueError("POLYGON_API_KEY environment variable not set")
OPENAI_API_KEY=os.getenv("OPENAI_API_KEY")
if OPENAI_API_KEY is None:
    raise ValueError("OPENAI_API_KEY environment variable not set")
STOCK_NAME = "Apple"
STOCK_SYMBOL = "AAPL"

polygon_client = RESTClient(API_KEY)
openai_client = OpenAI(api_key=OPENAI_API_KEY)

### Get Ticker

In [77]:

tickers = []
for t in polygon_client.list_tickers(
	ticker=STOCK_SYMBOL,
	market="stocks",
	active="true",
	order="asc",
	limit="100",
	sort="ticker",
	):
    tickers.append(t)

pprint(tickers)

[Ticker(active=True,
        cik='0000320193',
        composite_figi='BBG000B9XRY4',
        currency_name='usd',
        currency_symbol=None,
        base_currency_symbol=None,
        base_currency_name=None,
        delisted_utc=None,
        last_updated_utc='2025-04-30T00:00:00Z',
        locale='us',
        market='stocks',
        name='Apple Inc.',
        primary_exchange='XNAS',
        share_class_figi='BBG001S5N8V8',
        ticker='AAPL',
        type='CS',
        source_feed=None)]


### Get Daily Market Summary

#### get raw data

In [78]:
from datetime import timedelta

# Calculate the start date (30 days ago)
start_date = (date.today() - timedelta(days=45)).isoformat()

# Fetch the last 30 days of aggregate data
last_30_days = polygon_client.get_aggs(
    ticker=STOCK_SYMBOL,
    multiplier=1,
    timespan="day",
    from_=start_date,
    to=date.today().isoformat(),
    adjusted="true",
)

market_summaries_df = pd.DataFrame(last_30_days)
market_summaries_df.head()

,open,high,low,close,volume,vwap,timestamp,transactions,otc
0,213.31,215.2200,209.97,214.00,48073426.0,213.2536,1742184000000,577436,None
1,214.16,215.1500,211.49,212.69,42427021.0,213.0584,1742270400000,493002,None
2,214.22,218.7600,213.75,215.24,54383891.0,215.5599,1742356800000,524677,None
3,213.99,217.4899,212.22,214.10,48862947.0,214.3888,1742443200000,499769,None
4,211.56,218.8400,211.28,218.27,93568368.0,215.9292,1742529600000,496946,None


#### features

In [79]:
import pandas as pd
import numpy as np

def extract_timeseries_features(df: pd.DataFrame) -> dict:
    # 1. Ensure timestamp → datetime and sort
    df = df.copy()
    df['date'] = pd.to_datetime(df['timestamp'], unit='ms')
    df = df.sort_values('date').reset_index(drop=True)
    
    # 2. Compute daily returns
    df['return'] = df['close'].pct_change()
    
    # 3. Total and recent momentum
    total_return   = (df['close'].iloc[-1] / df['close'].iloc[0] - 1) * 100
    momentum_5d    = df['close'].pct_change(5).iloc[-1] * 100
    momentum_20d   = df['close'].pct_change(20).iloc[-1] * 100
    
    # 4. Volatility & drawdown
    vol_daily      = df['return'].std() * 100
    cum_returns    = (1 + df['return']).cumprod()
    max_drawdown   = (cum_returns / cum_returns.cummax() - 1).min() * 100
    
    # 5. Moving averages & crossover
    df['SMA_20']   = df['close'].rolling(20).mean()
    df['SMA_50']   = df['close'].rolling(50).mean()
    crossover      = 'bullish' if df['SMA_20'].iloc[-1] > df['SMA_50'].iloc[-1] else 'bearish'
    
    # 6. RSI(14) - plain 
    # delta          = df['close'].diff()
    # up, down       = delta.clip(lower=0), -delta.clip(upper=0)
    # roll_up        = up.rolling(14).mean()
    # roll_down      = down.rolling(14).mean()
    # rs             = roll_up / roll_down
    # df['RSI_14']   = 100 - (100 / (1 + rs))
    # rsi_latest     = df['RSI_14'].iloc[-1]

    # 6 RSI(14) using Wilder’s smoothing
    delta     = df['close'].diff()
    up        = delta.clip(lower=0)
    down      = -delta.clip(upper=0)
    avg_gain  = up.ewm(alpha=1/14, adjust=False).mean()
    avg_loss  = down.ewm(alpha=1/14, adjust=False).mean()
    rs        = avg_gain / avg_loss
    df['RSI_14'] = 100 - (100 / (1 + rs))
    rsi_latest  = df['RSI_14'].iloc[-1]
    
    # 7. Volume & VWAP signals
    avg_vol        = df['volume'].mean()
    vol_spike      = df['volume'].iloc[-1] / avg_vol
    df['vwap_dev'] = (df['close'] - df['vwap']) / df['vwap']
    vwap_dev_latest= df['vwap_dev'].iloc[-1] * 100
    
    # 8. Assemble feature dict
    features = {
        'total_return_%':     float(round(total_return, 2)),
        'momentum_5d_%':      float(round(momentum_5d, 2)),
        'momentum_20d_%':     float(round(momentum_20d, 2)),
        'volatility_daily_%': float(round(vol_daily, 2)),
        'max_drawdown_%':     float(round(max_drawdown, 2)),
        'SMA20_vs_SMA50':     crossover,
        'RSI_14':             float(round(rsi_latest, 2)),
        'avg_volume_m':       float(round(avg_vol / 1e6, 2)),
        'vol_spike_ratio':    float(round(vol_spike, 2)),
        'vwap_dev_%':         float(round(vwap_dev_latest, 2)),
    }
    return features

# — Example usage —
# assume your DataFrame is named `df`
sma_features = extract_timeseries_features(market_summaries_df)
pprint(sma_features)

{'RSI_14': 55.02,
 'SMA20_vs_SMA50': 'bearish',
 'avg_volume_m': 66.91,
 'max_drawdown_%': -22.99,
 'momentum_20d_%': -4.72,
 'momentum_5d_%': 2.38,
 'total_return_%': -0.32,
 'vol_spike_ratio': 0.86,
 'volatility_daily_%': 4.08,
 'vwap_dev_%': 0.68}


### Get Moving Averages

#### get data

In [80]:
sma = polygon_client.get_sma(
    ticker=STOCK_SYMBOL,
	timespan="day",
	adjusted="true",
	window="45",
	series_type="close",
	order="desc",
)

pprint(sma)


SingleIndicatorResults(values=[IndicatorValue(timestamp=1746072000000,
                                              value=213.2017777777778),
                               IndicatorValue(timestamp=1745985600000,
                                              value=213.80266666666674),
                               IndicatorValue(timestamp=1745899200000,
                                              value=214.5702222222223),
                               IndicatorValue(timestamp=1745812800000,
                                              value=215.3677777777779),
                               IndicatorValue(timestamp=1745553600000,
                                              value=216.15466666666677),
                               IndicatorValue(timestamp=1745467200000,
                                              value=216.96688888888897),
                               IndicatorValue(timestamp=1745380800000,
                                              value=217.778000000000

#### analyze

In [81]:
import pandas as pd
from datetime import datetime

def analyze_sma(sma_results, current_price):
    # 1. Build a DataFrame from the IndicatorValue list
    records = [
        {
            'date': pd.to_datetime(iv.timestamp, unit='ms'),
            'sma': float(iv.value)
        }
        for iv in sma_results.values
    ]
    df_sma = pd.DataFrame(records).sort_values('date').reset_index(drop=True)

    # 2. Compute trend metrics
    first_val   = df_sma['sma'].iloc[0]
    last_val    = df_sma['sma'].iloc[-1]
    periods     = len(df_sma)
    delta       = last_val - first_val
    pct_change  = (last_val / first_val - 1) * 100
    slope_per_p = delta / (periods - 1)               # absolute change per period
    pct_slope   = pct_change / (periods - 1)          # percent change per period

    # 3. Compare current price to SMA
    price_vs_sma_pct = (current_price / last_val - 1) * 100

    # 4. Round and cast to native Python types
    sma_features = {
        'sma_start_date':         df_sma['date'].iloc[0].strftime('%Y-%m-%d'),
        'sma_end_date':           df_sma['date'].iloc[-1].strftime('%Y-%m-%d'),
        'sma_periods':            int(periods),
        'sma_start_value':        float(round(first_val, 2)),
        'sma_end_value':          float(round(last_val, 2)),
        'sma_total_%_change':     float(round(pct_change, 2)),
        'sma_slope_per_period':   float(round(slope_per_p, 2)),
        'sma_pct_slope':          float(round(pct_slope, 2)),
        'price_vs_sma_%':         float(round(price_vs_sma_pct, 2)),
    }
    return sma_features

# — Example usage —
# `sma` is your SingleIndicatorResults; 
# `latest_close` you fetch separately (e.g. last Agg close)
sma_features = analyze_sma(sma, current_price=214.00)
pprint(sma_features)


{'price_vs_sma_%': 0.37,
 'sma_end_date': '2025-05-01',
 'sma_end_value': 213.2,
 'sma_pct_slope': -0.38,
 'sma_periods': 10,
 'sma_slope_per_period': -0.84,
 'sma_start_date': '2025-04-17',
 'sma_start_value': 220.74,
 'sma_total_%_change': -3.41}


### Get Financials

#### helpers

In [82]:
def make_summary_from_row(row):
    fs = row['financials']
    bs = fs.get('balance_sheet', {})
    is_ = fs.get('income_statement', {})
    cf = fs.get('cash_flow_statement', {})
    
    # helper to safely get a .value
    def V(section, key, default=0):
        return section.get(key, {}).get('value', default)
    
    # compute metrics
    metrics = {
        'Revenues':         V(is_, 'revenues') / 1e9,
        'Net Income':       V(is_, 'net_income_loss') / 1e9,
        'Current Ratio':    V(bs, 'current_assets') / V(bs, 'current_liabilities', 1),
        'Debt/Equity':      V(bs, 'liabilities') / V(bs, 'equity', 1),
        'Operating Margin': V(is_, 'operating_income_loss') / V(is_, 'revenues', 1),
        'Net Margin':       V(is_, 'net_income_loss') / V(is_, 'revenues', 1),
        'ROE':              V(is_, 'net_income_loss') / V(bs, 'equity', 1),
        # You could also add Free Cash Flow here:
        # 'Free Cash Flow': (V(cf,'net_cash_flow_from_operating_activities')
        #                    - estimated_capex) / 1e9
    }
    
    # format into lines
    lines = []
    for k, v in metrics.items():
        if k in ('Revenues', 'Net Income'):
            lines.append(f"{k}: ${v:.1f} B")
        else:
            lines.append(f"{k}: {v:.2%}")
    
    header = f"{row['company_name'][0]} {row['fiscal_period']} {row['fiscal_year']} Summary:"
    return header + "\n• " + "\n• ".join(lines)

#### summary text

In [83]:
financials = []
for f in polygon_client.vx.list_stock_financials(
	ticker=STOCK_SYMBOL,
	filing_date_gte="2024-01-01",
	order="asc",
	limit="10",
	sort="filing_date",
	):
    financials.append(f)
financials_df = pd.DataFrame(financials)
financials_df['filing_date'] = pd.to_datetime(financials_df['filing_date'])
latest = financials_df.sort_values('filing_date').iloc[-1]

summary_text = make_summary_from_row(latest)
print(summary_text)

A Q1 2025 Summary:
• Revenues: $124.3 B
• Net Income: $36.3 B
• Current Ratio: 92.29%
• Debt/Equity: 415.42%
• Operating Margin: 34.46%
• Net Margin: 29.23%
• ROE: 54.42%


### Get News Sentiments

#### get data

In [84]:
from polygon import RESTClient
from polygon.rest.models import (
    TickerNews,
)

start_date = (datetime.now() - timedelta(days=45))

news = []
for n in polygon_client.list_ticker_news(
    ticker=STOCK_SYMBOL,
    order="asc",
    sort="published_utc",
    published_utc_gte=start_date.isoformat() + "Z",
    ):
    news.append(n)

last_5_news = news[-5:]

ticker_news_summary = ""
for last_news in last_5_news:
    ticker_news_summary += f"""
    article url: {last_news.article_url}
    description: {last_news.description}
    ---
    """

print(ticker_news_summary)






    article url: https://www.fool.com/investing/2025/05/01/down-nearly-20-this-ai-giant-is-the-best-bargain-m/?source=iedfolrf0000001
    description: Nvidia, one of the 'Magnificent Seven' tech giants, has seen its stock drop nearly 20% this year due to concerns over potential import tariffs and restrictions on exports to China. However, the article argues that Nvidia's strong growth prospects, market leadership, and efforts to limit tariff impact make it the best bargain among the 'Magnificent Seven' stocks.
    ---
    
    article url: https://www.investing.com/analysis/nasdaq-100-hits-resistance-near-20000-after-microsoft-meta-blowout-results-200660229
    description: The Nasdaq 100 index is nearing the 20,000 level after strong earnings results from tech giants Microsoft and Meta. The improved sentiment and hopes of a US-China trade deal have set up the upcoming earnings reports from Amazon and Apple to have a notable impact.
    ---
    
    article url: https://www.investing.

### Prompt LLM

In [85]:
response = openai_client.responses.create(
    model="gpt-4.1",
    input=f"""
    Below are {STOCK_NAME} stock metrics:

    Quarterly Financials:
    {summary_text}

    Last 45 day metrics:    
    {sma_features}

    45 day SMA:
    {sma_features}

    Recent News:
    {ticker_news_summary}

    Please analyze these and output **only** a JSON object with this exact schema:

    {{
    "score": <integer 0-100, where 0 = very weak, 100 = very strong>,
    "recommendation": "<BUY, HOLD, or SELL>",
    "confidence": <float 0.0-1.0>,
    "rationale": "<one-sentence explanation>"
    }}

    Do not output any extra text."""
)

print(response.output_text)

{
"score": 70,
"recommendation": "HOLD",
"confidence": 0.75,
"rationale": "Apple's financials remain strong with high margins and cash returns, but the negative short-term technical trend and ongoing China trade risks justify caution."
}
